# Module 02 · Unit 01
# Predicates and Quantifiers

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] · Protocol Verification \[PV\] |
| **Refresher** | Module 01 · Unit 01–03 — Propositional Logic & Boolean Algebra |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Distinguish a **predicate** from a proposition, and explain why the distinction matters
2. Write and evaluate **universally** ($\forall$) and **existentially** ($\exists$) quantified statements
3. Identify **free** and **bound** variables in a quantified formula
4. Correctly **negate** quantified statements using the quantifier duality laws
5. Interpret **nested quantifiers** and explain why order matters
6. Use Python to evaluate predicates over finite domains and search for witnesses and counterexamples


## 🔣 Symbol Primer

Module 02 promotes $\forall$ and $\exists$ — introduced briefly in Module 00 —
to first-class objects. Four new notations appear alongside them.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $P(x)$ | Predicate | "$P$ of $x$" | A property that depends on $x$ — becomes a proposition when $x$ is specified |
| $\forall x \in D,\; P(x)$ | Universal quantifier | "for all $x$ in $D$, $P(x)$" | $P$ holds for **every** element of domain $D$ |
| $\exists x \in D,\; P(x)$ | Existential quantifier | "there exists $x$ in $D$ such that $P(x)$" | $P$ holds for **at least one** element of $D$ |
| $\exists!\, x \in D,\; P(x)$ | Unique existential | "there exists a **unique** $x$ in $D$ such that $P(x)$" | Exactly **one** element of $D$ satisfies $P$ |
| $\mid$ | "such that" | (in set-builder notation) | Separates the variable from the condition: $\{x \in D \mid P(x)\}$ |

> **The key upgrade from Module 01.** Propositional logic talks about fixed
> truth values: $A$ is either true or false, period. Predicate logic talks about
> *properties of objects*: $\text{admin}(u)$ might be true for user Alice and
> false for user Bob. Quantifiers let us make claims about **all** objects or
> **some** objects in a domain — the vocabulary of every real security property.

---


## 1 · From Propositions to Predicates

In Module 01 we wrote $A$ to mean "the user is an admin." That was fine for a
single decision, but it hardcoded one user. Real systems have thousands of users,
millions of inputs, billions of memory addresses. We need a way to talk about
properties that *vary* depending on what object we are looking at.

A **predicate** is a function from objects to truth values. It becomes a
proposition the moment you plug in a specific object.

### Examples

| Predicate | Written | Becomes a proposition when... |
|---|---|---|
| "$x$ is an admin" | $\text{admin}(x)$ | $x = \text{Alice}$ → $\text{admin}(\text{Alice})$ |
| "$n$ is prime" | $\text{prime}(n)$ | $n = 7$ → $\text{prime}(7)$ is True |
| "$s$ contains a SQL keyword" | $\text{injectable}(s)$ | $s = \texttt{' OR 1=1}$ → $\text{injectable}(s)$ is True |
| "$p$ is within allocation $a$" | $\text{in\_bounds}(p, a)$ | $p = 0\text{x}1000,\; a = [0\text{x}0800,\,0\text{x}2000]$ |

The last example has **two** arguments — predicates can take any number. A
two-argument predicate $R(x, y)$ is called a **relation**, which we will study
formally in Module 03.

### The Domain of Discourse

Every predicate quantifier ranges over a **domain** $D$ — the set of objects
the variable is allowed to take. This must always be stated explicitly.

$$\forall x \in D, \quad P(x)$$

Without naming $D$, the statement is incomplete. The choice of domain changes
the truth value: $\forall n \in \{2, 3, 5, 7\},\; \text{prime}(n)$ is True,
but $\forall n \in \mathbb{Z},\; \text{prime}(n)$ is False (9 is a counterexample).


## 2 · The Universal Quantifier — $\forall$

$\forall x \in D,\; P(x)$ *(for all $x$ in $D$, $P(x)$)* is True if and only if
$P(x)$ holds for **every single element** of $D$. One exception makes it False.

### Evaluating $\forall$ Over a Finite Domain

Over a finite domain $D = \{d_1, d_2, \ldots, d_n\}$:

$$\forall x \in D,\; P(x) \;\equiv\; P(d_1) \wedge P(d_2) \wedge \cdots \wedge P(d_n)$$

The universal quantifier is a **giant conjunction**. This is why finding one
counterexample suffices to disprove it — one False in a conjunction makes the
whole thing False.

### Security Examples

$$\forall u \in \text{Users},\; \text{has\_role}(u) \quad
\text{"Every user has been assigned at least one role."}$$

$$\forall s \in \text{Inputs},\; \text{validate}(s) = \mathtt{true} \Rightarrow \text{safe}(s)$$

$$\quad \text{"Every input that passes validation is safe to process."}$$

$$\forall p \in \text{Processes},\; \text{privilege}(p) \leq \text{MAX\_PRIV}$$

$$\quad \text{"No process exceeds the maximum privilege level."}$$

The third statement is exactly the **invariant** from Module 00, now written in
predicate logic. Invariants are universal claims over the set of all reachable states.

### Proof and Disproof

- To **prove** $\forall x \in D,\; P(x)$: argue that $P(x)$ holds for an arbitrary,
  unspecified $x$ — do not assume anything about $x$ beyond it being in $D$.
- To **disprove** $\forall x \in D,\; P(x)$: exhibit one specific $x_0 \in D$
  with $\neg P(x_0)$. This is a **counterexample** (Module 00).


## 3 · The Existential Quantifier — $\exists$

$\exists x \in D,\; P(x)$ *(there exists $x$ in $D$ such that $P(x)$)* is True
if and only if $P(x)$ holds for **at least one** element of $D$.

### Evaluating $\exists$ Over a Finite Domain

$$\exists x \in D,\; P(x) \;\equiv\; P(d_1) \vee P(d_2) \vee \cdots \vee P(d_n)$$

The existential quantifier is a **giant disjunction**. One True element is enough.

### Security Examples

$$\exists s \in \text{Inputs},\; \text{injectable}(s) \quad
\text{"There exists at least one injectable input."}$$

This is the existence of a vulnerability — the attacker's goal. Proving this claim
requires finding *one* input, which is exactly what a penetration test does.

$$\exists u \in \text{Users},\; \text{admin}(u) \wedge \text{suspended}(u)$$

$$\quad \text{"There is at least one suspended admin account."}$$

A security audit checking for suspended privileged accounts is an $\exists$ search
over the user database.

### The Witness

The specific element $x_0$ that makes $P(x_0)$ true is called a **witness**. To
prove an existential claim, you produce a witness. To disprove a universal claim,
you produce a witness to the negation — a counterexample.

$$\text{Witness for } \exists x,\; P(x): \quad \text{show one } x_0 \text{ with } P(x_0) = \mathtt{true}$$
$$\text{Counterexample for } \forall x,\; P(x): \quad \text{show one } x_0 \text{ with } P(x_0) = \mathtt{false}$$

Both are the same operation — finding a specific element. The difference is only
which claim you are evaluating.


## 4 · Negating Quantified Statements — Quantifier Duality

Negating a quantified statement flips the quantifier **and** negates the predicate.
These are the quantifier analogues of De Morgan's Laws:

$$\neg\,\bigl(\forall x \in D,\; P(x)\bigr) \;\equiv\; \exists x \in D,\; \neg P(x)$$

$$\neg\,\bigl(\exists x \in D,\; P(x)\bigr) \;\equiv\; \forall x \in D,\; \neg P(x)$$

**How to remember:** negation pushes through a quantifier and flips it.
$\forall$ becomes $\exists$, $\exists$ becomes $\forall$, and $P$ becomes $\neg P$.

### Why This Matters — Computing Violation Conditions

Every security property is stated as a positive claim. The **violation** of that
claim is its negation — and the negation is what an attacker is trying to achieve,
and what a monitoring system is trying to detect.

| Security property | Formal statement | Violation (negation) |
|---|---|---|
| All inputs validated | $\forall s,\; \text{validate}(s)$ | $\exists s,\; \neg\,\text{validate}(s)$ |
| No suspended admins | $\neg\,\exists u,\; \text{admin}(u) \wedge \text{suspended}(u)$ | $\exists u,\; \text{admin}(u) \wedge \text{suspended}(u)$ |
| Every process in bounds | $\forall p,\; \text{in\_bounds}(p)$ | $\exists p,\; \neg\,\text{in\_bounds}(p)$ |

The violation of a $\forall$ property is always an $\exists$ claim — find one
bad element. This is the formal statement of the adversarial asymmetry from Module 00.

### Worked Example

**Property:** "Every API endpoint requires authentication."

$$\forall e \in \text{Endpoints},\; \text{requires\_auth}(e)$$

**Violation** (push $\neg$ through $\forall$, flip to $\exists$, negate predicate):

$$\exists e \in \text{Endpoints},\; \neg\,\text{requires\_auth}(e)$$

*"There exists at least one endpoint that does not require authentication."*

This is exactly the finding format of an API security audit.


## 5 · Free and Bound Variables

In the formula $\forall x \in D,\; P(x)$, the variable $x$ is **bound** by the
quantifier — it ranges over $D$ and has no independent meaning outside the formula.

A variable that appears in a formula but is **not** bound by any quantifier is
called **free**. A formula with free variables is a predicate — not yet a
proposition. It becomes a proposition only when the free variable is substituted
with a specific value or bound by a quantifier.

### Examples

| Formula | Free variables | Bound variables | Is it a proposition? |
|---|---|---|---|
| $\text{admin}(x)$ | $x$ | none | No — depends on $x$ |
| $\forall x \in U,\; \text{admin}(x)$ | none | $x$ | Yes |
| $\text{admin}(x) \wedge \text{active}(y)$ | $x$, $y$ | none | No |
| $\exists x \in U,\; \text{admin}(x) \wedge \text{active}(y)$ | $y$ | $x$ | No — $y$ still free |
| $\forall y \in U,\; \exists x \in U,\; \text{admin}(x) \wedge \text{active}(y)$ | none | $x$, $y$ | Yes |

### Why This Matters in Practice

A common error in policy specification is writing a formula with a free variable
and treating it as if it were universally quantified. For example:

$$\text{validate}(s) \Rightarrow \text{safe}(s)$$

This *looks* like a policy but is actually a predicate — it is True or False
depending on what $s$ is. The actual policy claim requires explicit quantification:

$$\forall s \in \text{Inputs},\; \text{validate}(s) \Rightarrow \text{safe}(s)$$

Without the $\forall$, the implication holds for one specific (but unspecified) $s$.
The universal quantifier is what makes it a security guarantee over all inputs.


## 6 · Nested Quantifiers — Order Matters

Quantifiers can be nested. The order is critical — $\forall x\, \exists y$ and
$\exists y\, \forall x$ mean completely different things.

### The Key Distinction

$$\forall x \in D,\; \exists y \in D,\; R(x, y)$$

*"For every $x$, there exists some $y$ (possibly depending on $x$) such that $R(x, y)$."*
Each $x$ may have a **different** $y$.

$$\exists y \in D,\; \forall x \in D,\; R(x, y)$$

*"There exists a single $y$ that works for **all** $x$ simultaneously."*
One fixed $y$ must serve every $x$.

The second statement is strictly stronger. If $\exists y\,\forall x\, R(x,y)$ is
true, then $\forall x\,\exists y\, R(x,y)$ is automatically true — but not vice versa.

### Security Illustration — Key Management

Let $\text{can\_decrypt}(u, k)$ mean "user $u$ can use key $k$ to decrypt a message."

$$\forall u \in \text{Users},\; \exists k \in \text{Keys},\; \text{can\_decrypt}(u, k)$$

*"Every user has at least one decryption key — possibly a different key per user."*
This is the expected state of a key management system: each user has their own key.

$$\exists k \in \text{Keys},\; \forall u \in \text{Users},\; \text{can\_decrypt}(u, k)$$

*"There exists a single key that every user can use."*
This is a **catastrophic key management failure** — one shared key means that
compromising any user compromises all encrypted data.

The difference between these two statements is the difference between a well-designed
and a catastrophically broken cryptographic system. The quantifier order encodes it.

### Negating Nested Quantifiers

Apply duality repeatedly, working from the outside in:

$$\neg\,(\forall x,\; \exists y,\; P(x,y))
  \;\equiv\; \exists x,\; \neg\,(\exists y,\; P(x,y))
  \;\equiv\; \exists x,\; \forall y,\; \neg P(x,y)$$

Each negation pushes one level deeper, flipping each quantifier it passes through.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\] \[PV\]

| Concept | Security meaning | Example |
|---|---|---|
| **Predicate** | A testable property of a system object | $\text{injectable}(s)$, $\text{admin}(u)$, $\text{in\_bounds}(p)$ |
| **$\forall$ quantifier** | A security guarantee — must hold for every object | "Every input is validated" |
| **$\exists$ quantifier** | A vulnerability or audit finding — one bad object exists | "There is an unauthenticated endpoint" |
| **Witness** | A proof of $\exists$ — the specific bad object | The PoC input in a CVE |
| **Negation duality** | Flip $\forall \leftrightarrow \exists$, negate predicate | Convert a guarantee to its violation condition |
| **Free variable** | An unquantified claim — not yet a guarantee | A policy template that names no specific domain |
| **Nested $\forall\exists$** | Each user has their own key — correct design | Per-user key management |
| **Nested $\exists\forall$** | One key shared by all users — catastrophic | Single shared secret |

**The practitioner habit.** Every time you read a security claim, ask:
- What is the **domain**? (All inputs? All users? All network packets?)
- Is this $\forall$ or $\exists$? (Guarantee or existence-of-vulnerability?)
- Are there **free variables** — things the claim depends on but doesn't quantify?
- What does the **negation** look like — what would a violation look like?

Answering these four questions turns vague security English into a formal claim
that can be tested, proved, or refuted.

---


## Python: Visualization & Verification

**1 · Predicate Evaluator** — define predicates as Python functions over a finite
domain, evaluate them exhaustively, and visualise the satisfying set.

**2 · Quantifier Search** — implement $\forall$ and $\exists$ evaluation with
explicit witness and counterexample extraction; demonstrate negation duality.

**3 · Nested Quantifier Explorer** — compare $\forall x\,\exists y$ against
$\exists y\,\forall x$ on the key management example to make the difference concrete.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import itertools
import random

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Predicate Evaluator

A predicate over a finite domain is just a Python function that returns a bool.
We define three security-flavoured predicates over a small simulated user database
and visualise which users satisfy each one — the **extension** of the predicate.

The extension of $P$ over domain $D$ is the set $\{x \in D \mid P(x)\}$ —
every element that makes $P$ true. Visualising the extension makes the
$\forall$ / $\exists$ structure immediately concrete.


In [ ]:
# ── 1 · Predicate Evaluator and Extension Visualizer ─────────────────────────
random.seed(7)

# Simulated user database — 12 users, each with attributes
users = [
    {'id': f'u{i:02d}', 'name': name,
     'admin': admin, 'active': active, 'mfa': mfa, 'suspended': susp}
    for i, (name, admin, active, mfa, susp) in enumerate([
        ('Alice',   True,  True,  True,  False),
        ('Bob',     True,  True,  False, False),   # admin, no MFA
        ('Carol',   False, True,  True,  False),
        ('Dave',    False, True,  False, False),
        ('Eve',     True,  False, True,  False),   # admin, inactive
        ('Frank',   False, False, False, False),
        ('Grace',   True,  True,  True,  True),    # admin, suspended
        ('Heidi',   False, True,  True,  False),
        ('Ivan',    True,  True,  True,  False),
        ('Judy',    False, True,  False, True),    # suspended, non-admin
        ('Karl',    True,  True,  True,  False),
        ('Laura',   False, True,  True,  False),
    ], start=0)
]

# Predicates
def admin(u):      return u['admin']
def active(u):     return u['active']
def mfa(u):        return u['mfa']
def suspended(u):  return u['suspended']
def safe_admin(u): return u['admin'] and u['active'] and u['mfa'] and not u['suspended']

predicates = {
    r'$\mathrm{admin}(u)$':       admin,
    r'$\mathrm{active}(u)$':      active,
    r'$\mathrm{mfa}(u)$':         mfa,
    r'$\mathrm{suspended}(u)$':   suspended,
    r'$\mathrm{safe\_admin}(u)$': safe_admin,
}

# ── Heatmap — each cell shows whether user u satisfies predicate P ─────────────
fig, ax = plt.subplots(figsize=(12, 5))

names    = [u['name'] for u in users]
pred_lbl = list(predicates.keys())
matrix   = np.array([[float(f(u)) for u in users] for f in predicates.values()])

cmap = ListedColormap([TS_RED, TS_GREEN])
ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

for r, plbl in enumerate(pred_lbl):
    for c, u in enumerate(users):
        val = bool(matrix[r, c])
        ax.text(c, r, 'T' if val else 'F', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(pred_lbl)))
ax.set_yticklabels(pred_lbl, fontsize=10)
ax.set_title('Predicate Extensions over User Domain'
             'Green = True (user satisfies the predicate), Red = False', pad=12)

handles = [mpatches.Patch(color=TS_GREEN, label='Satisfies P(u)'),
           mpatches.Patch(color=TS_RED,   label='Does not satisfy P(u)')]
ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Print extension sizes
print("Extension sizes  |{x ∈ Users | P(x)}|:")
for lbl, f in predicates.items():
    ext = [u['name'] for u in users if f(u)]
    print(f"  {lbl.replace('$','').replace('\\',''):<25} {len(ext):>2} users: {', '.join(ext)}")


**What do you see?**

Each row is the **extension** of one predicate — the set of users that satisfy it.
A few things to notice immediately:

- $\text{safe\_admin}(u)$ — the bottom row — has very few green cells. This is the
  conjunction of four conditions; each extra clause eliminates more users.
  Only admins who are active, MFA-enrolled, and not suspended pass.
- Bob is admin and active but has no MFA — he fails $\text{safe\_admin}$.
- Grace is admin, active, and MFA-enrolled but is suspended — she also fails.
- These are not hypothetical gaps; they are the real attack surface.

Look at the heatmap and ask: is $\forall u,\; \text{mfa}(u)$ true over this domain?
Is $\exists u,\; \text{suspended}(u) \wedge \text{admin}(u)$ true?
You can read both answers directly from the grid.


### 2 · Quantifier Search — Witnesses and Counterexamples

We now implement universal and existential evaluation explicitly, with witness
and counterexample extraction. The functions make the relationship between
quantifiers and iteration precise: $\forall$ is `all(...)`, $\exists$ is `any(...)`.

We also verify negation duality: the violation of every $\forall$ claim we test
is found by flipping to $\exists$ and negating the predicate.


In [ ]:
# ── 2 · Quantifier Search with Witness / Counterexample Extraction ────────────

def forall(domain, predicate):
    """Evaluate ∀x ∈ domain, P(x).
    Returns (result, counterexample_or_None).
    """
    for x in domain:
        if not predicate(x):
            return False, x
    return True, None

def exists(domain, predicate):
    """Evaluate ∃x ∈ domain, P(x).
    Returns (result, witness_or_None).
    """
    for x in domain:
        if predicate(x):
            return True, x
    return False, None

# ── Security claims to evaluate ───────────────────────────────────────────────
claims = [
    ("∀u, active(u)",
     lambda: forall(users, active),
     "Every user is active"),

    ("∀u, mfa(u)",
     lambda: forall(users, mfa),
     "Every user has MFA enrolled"),

    ("∀u, admin(u) → mfa(u)",
     lambda: forall(users, lambda u: not admin(u) or mfa(u)),
     "Every admin has MFA (if admin then MFA)"),

    ("∀u, ¬(admin(u) ∧ suspended(u))",
     lambda: forall(users, lambda u: not (admin(u) and suspended(u))),
     "No admin is suspended"),

    ("∃u, admin(u) ∧ ¬mfa(u)",
     lambda: exists(users, lambda u: admin(u) and not mfa(u)),
     "There is an admin without MFA"),

    ("∃u, suspended(u) ∧ admin(u)",
     lambda: exists(users, lambda u: suspended(u) and admin(u)),
     "There is a suspended admin"),
]

print(f"{'Claim':<40} {'Result':<8} {'Witness / Counterexample'}")
print("─" * 78)
for formula, evaluate, description in claims:
    result, evidence = evaluate()
    if evidence:
        ev_str = evidence['name']
        ev_type = 'witness' if result else 'counterexample'
    else:
        ev_str  = '—'
        ev_type = ''
    result_str = 'TRUE ' if result else 'FALSE'
    flag = '✓' if result else '✗'
    print(f"  {flag}  {formula:<36} {result_str:<8} "
          f"{ev_str} {('(' + ev_type + ')') if ev_type else ''}")

# ── Negation duality check ─────────────────────────────────────────────────────
print()
print("─" * 78)
print("Negation duality verification:")
print("  ¬(∀u, admin(u) → mfa(u))  should equal  ∃u, admin(u) ∧ ¬mfa(u)")

r1, cx = forall(users, lambda u: not admin(u) or mfa(u))
r2, w  = exists(users, lambda u: admin(u) and not mfa(u))

print(f"  ¬(∀ claim) = ¬{r1} = {not r1}")
print(f"   ∃ claim   = {r2}")
print(f"  Match: {not r1 == r2}  — duality law holds ✓" if (not r1)==r2
      else "  Mismatch — something is wrong")
if cx:
    print(f"  Counterexample for ∀ = witness for ∃: {cx['name']}")


**What do you see?**

- "Every user has MFA" is **False** — Bob is the counterexample.
- "Every admin has MFA" is **False** — again Bob (admin, no MFA).
- "No admin is suspended" is **False** — Grace is the counterexample.
- The two existential claims are **True** — Bob (admin, no MFA) and Grace
  (suspended admin) are their witnesses.

The negation duality check at the bottom confirms that the $\forall$ claim
and its negated $\exists$ form give opposite truth values — and the counterexample
to the $\forall$ claim is exactly the witness for the $\exists$ claim.

**This is the formal shape of a security audit.** The claims are the policy
requirements. The counterexamples are the findings. `forall` and `exists` are
the audit queries. The user database is the domain.


### 3 · Nested Quantifier Explorer — Key Management

We now make the $\forall x\,\exists y$ vs $\exists y\,\forall x$ distinction
concrete using the key management scenario from Section 6.

**Domain:** 6 users, 6 keys. Each user–key pair either grants decryption access
or does not. We test two models:

- **Model A (correct):** Each user has their *own* unique key — no sharing.
  $\forall u\,\exists k,\; \text{can\_decrypt}(u, k)$ is True, but
  $\exists k\,\forall u,\; \text{can\_decrypt}(u, k)$ is False.

- **Model B (broken):** One master key decrypts everything — everyone shares it.
  Both statements are True, but the $\exists k\,\forall u$ is the catastrophic one.

The heatmap shows the access matrix for each model.


In [ ]:
# ── 3 · Nested Quantifier Explorer — Key Management ─────────────────────────
random.seed(42)

user_names = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank']
key_names  = [f'k{i}' for i in range(6)]

# Model A: each user has exactly one personal key (diagonal)
access_A = np.eye(6, dtype=bool)

# Model B: key k0 is a master key — everyone can use it
access_B = np.zeros((6, 6), dtype=bool)
for i in range(6):
    access_B[i, 0] = True        # shared master key k0
    access_B[i, i] = True        # plus personal key

def check_nested(access_matrix, users, keys):
    """
    Evaluate:
      forall_exists: ∀u ∃k  can_decrypt(u,k)
      exists_forall: ∃k ∀u  can_decrypt(u,k)
    Returns (forall_exists, exists_forall, shared_key_or_None)
    """
    fe = all(any(access_matrix[u, k] for k in range(len(keys)))
             for u in range(len(users)))
    ef_key = None
    for k in range(len(keys)):
        if all(access_matrix[u, k] for u in range(len(users))):
            ef_key = keys[k]
            break
    ef = ef_key is not None
    return fe, ef, ef_key

fe_A, ef_A, shared_A = check_nested(access_A, user_names, key_names)
fe_B, ef_B, shared_B = check_nested(access_B, user_names, key_names)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cmap = ListedColormap([TS_LIGHT, TS_GREEN])

for ax, matrix, title, fe, ef, shared in [
    (axes[0], access_A, 'Model A — Per-User Keys (Correct design)', fe_A, ef_A, shared_A),
    (axes[1], access_B, 'Model B — Shared Master Key (Catastrophic design)', fe_B, ef_B, shared_B),
]:
    ax.imshow(matrix.astype(float), cmap=cmap, vmin=0, vmax=1, aspect='auto')

    for r in range(6):
        for c in range(6):
            val = matrix[r, c]
            ax.text(c, r, '✓' if val else '·',
                    ha='center', va='center',
                    fontsize=14 if val else 10,
                    fontweight='bold' if val else 'normal',
                    color=TS_BLUE if val else TS_GRAY)

    ax.set_xticks(range(6))
    ax.set_xticklabels(key_names, fontsize=10)
    ax.set_yticks(range(6))
    ax.set_yticklabels(user_names, fontsize=10)
    ax.set_xlabel('Key')
    ax.set_ylabel('User')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=10)

    fe_sym = '✓ TRUE' if fe else '✗ FALSE'
    ef_sym = '✓ TRUE' if ef else '✗ FALSE'
    fe_col = TS_GREEN if fe else TS_RED
    ef_col = TS_GREEN if ef else TS_RED

    ax.text(0.02, -0.18,
            f'∀u ∃k can_decrypt(u,k):  {fe_sym}',
            transform=ax.transAxes, fontsize=9,
            color=fe_col, fontweight='bold')
    ax.text(0.02, -0.26,
            f'∃k ∀u can_decrypt(u,k):  {ef_sym}'
            + (f'  → shared key = {shared}' if shared else ''),
            transform=ax.transAxes, fontsize=9,
            color=ef_col, fontweight='bold')

plt.suptitle('Nested Quantifiers: ∀u ∃k  vs  ∃k ∀u'
             ' — Same access matrix structure, very different security properties',
             fontsize=11, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

print("Model A: ∀u ∃k — TRUE (each user has a personal key)")
print("         ∃k ∀u — FALSE (no single key works for all users)")
print("         → Correct design: compromise of one key affects one user")
print()
print("Model B: ∀u ∃k — TRUE  (each user still has a personal key too)")
print("         ∃k ∀u — TRUE   (k0 is a master key for all users)")
print(f"         → Catastrophic: compromising {shared_B} exposes ALL users' data")


**What do you see?**

Model A has one ✓ per row — each user has exactly one key. The $\forall u\,\exists k$
claim is True (every user has a key), but $\exists k\,\forall u$ is False (no key
unlocks everything). Compromise of any one key exposes exactly one user.

Model B adds a shared master key $k_0$ (the first column, all green). Now
$\exists k\,\forall u$ is **True** — $k_0$ is the witness. A single stolen key
compromises every user's data simultaneously.

Both models satisfy $\forall u\,\exists k$ — you cannot tell them apart from that
weaker property alone. The stronger $\exists k\,\forall u$ is what distinguishes
a safe design from a catastrophic one. **Quantifier order is a security property.**

**The formal habit to build:** whenever you write or review a key management or
credential design, ask: *does $\exists k\,\forall u$ hold?* If the answer is yes,
there is a single point of catastrophic failure.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. UNIQUE EXISTENTIAL
#    The unique existential ∃! means exactly one witness exists.
#    Implement:
#      unique_exists(domain, predicate) → (bool, list_of_witnesses)
#    Apply it to: ∃! u ∈ Users, admin(u) ∧ safe_admin(u) ∧ name starts with 'A'
#    How many users satisfy safe_admin? Is there exactly one admin named Alice?
#
# 2. MULTI-ARGUMENT PREDICATE
#    Define a two-argument predicate:
#      can_access(user, resource)
#    where resources = ['payroll', 'source_code', 'logs', 'public_docs']
#    and access rules are: admin → all; active non-admin → source_code, logs, public_docs
#    Build the full access matrix and evaluate:
#      ∀r ∃u can_access(u, r)   (every resource has at least one accessor)
#      ∀u ∃r can_access(u, r)   (every user can access at least one resource)
#      ∃u ∀r can_access(u, r)   (some user has universal access)
#
# 3. NEGATION CHAIN
#    Negate the following step by step, citing the duality law at each step:
#      ∀u ∈ Users, ∃s ∈ Sessions, ∀r ∈ Requests, authenticated(u, s, r)
#    What does the negated statement mean in plain English?
#    What would a monitoring system need to detect to catch a violation?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Predicate** $P(x)$ | Property depending on $x$ | A testable condition on a system object |
| **Domain** $D$ | The set $x$ ranges over | Users, inputs, processes, keys... |
| **$\forall x \in D,\; P(x)$** | $P$ holds for every element | A security guarantee — must hold universally |
| **$\exists x \in D,\; P(x)$** | $P$ holds for at least one element | A vulnerability — one bad object exists |
| **Witness** | An $x_0$ with $P(x_0)$ true | The specific finding in a security audit |
| **Counterexample** | An $x_0$ with $P(x_0)$ false | Proof that a $\forall$ guarantee fails |
| **Negation duality** | $\neg\forall \equiv \exists\neg$, $\neg\exists \equiv \forall\neg$ | Guarantee ↔ violation condition |
| **Free variable** | Unbound — formula is still a predicate | An incomplete policy specification |
| **$\forall x\,\exists y$** | Each $x$ gets its own $y$ | Per-user keys — correct design |
| **$\exists y\,\forall x$** | One $y$ works for all $x$ | Shared master key — catastrophic |

---

## Up Next

**Module 02 · Unit 02 — Formalizing Security Properties**

We put the full predicate logic toolkit to work. We write formal specifications
for injection safety, privilege bounds, session integrity, and protocol correctness —
then use Python to evaluate each specification against a simulated system and
generate audit reports in the format of a real security review.

$\rightarrow$ `modules/module-02/unit-02-formalizing-security-properties.ipynb`
